### Google Colab Setup

If running in Google Colab, run the cell below first to set up the environment.
If running locally (Jupyter), skip this cell.


In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Mount Drive and navigate to repo
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        # Generate data if needed
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        # Try cloned path
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')


# Case Study: Handling Imbalanced Classes

Real-world classification problems often have imbalanced target distributions.
This notebook explores techniques to deal with class imbalance: resampling, algorithmic adjustments, and evaluation metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_recall_curve
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
%matplotlib inline
print('Libraries loaded successfully!')


## 1. Load & Explore Data


In [ ]:
df = pd.read_csv('Data/fraud_data.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
# Check class distribution
df['is_fraud'].value_counts(normalize=True).mul(100).round(2)


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='is_fraud')
plt.title('Class Distribution (0 = Normal, 1 = Fraud)')
plt.show()


## 2. Train a Baseline Model


In [ ]:
X = df.drop(['transaction_id', 'is_fraud'], axis=1)
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)
y_proba = rf.predict_proba(X_test_scaled)[:, 1]

print('=== Baseline Model ===')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')


## 3. Addressing Imbalance with SMOTE


In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:', np.bincount(y_train))
print('After SMOTE:', np.bincount(y_train_res))


In [ ]:
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_res, y_train_res)
y_pred_smote = rf_smote.predict(X_test_scaled)
y_proba_smote = rf_smote.predict_proba(X_test_scaled)[:, 1]

print('=== After SMOTE ===')
print(classification_report(y_test, y_pred_smote))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_smote):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_smote):.4f}')


## 4. Undersampling


In [ ]:
under = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under.fit_resample(X_train_scaled, y_train)

rf_under = RandomForestClassifier(n_estimators=100, random_state=42)
rf_under.fit(X_train_under, y_train_under)
y_pred_under = rf_under.predict(X_test_scaled)

print('=== After Undersampling ===')
print(classification_report(y_test, y_pred_under))
print(f'F1 Score: {f1_score(y_test, y_pred_under):.4f}')


## 5. Comparison & Discussion


In [ ]:
results = pd.DataFrame({
    'Method': ['Baseline', 'SMOTE', 'Undersampling'],
    'F1 Score': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_smote),
        f1_score(y_test, y_pred_under)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_smote),
        roc_auc_score(y_test, rf_under.predict_proba(X_test_scaled)[:, 1])
    ]
})
results


## Summary


In [ ]:
print('''
Key Takeaways:
- Baseline models ignore minority class → poor recall for fraud class
- SMOTE creates synthetic minority samples → better recall without losing majority data
- Undersampling reduces majority class → fast but loses information
- Always use metrics like F1, ROC-AUC, Precision-Recall (not just accuracy)
''')
